# 52) Power Analysis (Test Gücü Analizi) / Örneklem Büyüklüğü
Konu (Tip I ve Tip II Hatalar)'da **Test Gücü**'nü ($1-\beta$) tanımlamıştık: "testin, gerçekte var olan bir etkiyi yakalama kapasitesi." Power Analysis, bu kavramı **pratiğe döken**, "bir testi güvenilir yapmak için kaç kişiye/gözleme ihtiyacım var?" sorusuna cevap veren tekniktir.

## Neden Önemli?
Bir A/B testi tasarlarken, testi başlatmadan ÖNCE şu soruyu sormalıyız: **"Eğer gerçekten bir etki varsa, benim örneklem büyüklüğüm bu etkiyi yakalayacak kadar yeterli mi?"** Yetersiz örneklem büyüklüğü ile yapılan bir test, gerçek bir etkiyi bile "istatistiksel olarak anlamlı değil" gösterebilir bu, **Tip II Hata riskini** artırır (Konu 49).

## Power Analysis'in Dört Bileşeni
Bu dört değerden **herhangi üçünü** bilirsen, dördüncüsünü hesaplayabilirsin birbirine matematiksel olarak bağlıdırlar:

1. **α (Anlamlılık Düzeyi):** Genelde 0.05 (Konu - Anlamlılık Düzeyi)
2. **Power (1-β):** Genelde 0.80 (%80) hedeflenir "gerçek bir etki varsa, bunu yakalama şansımın %80 olmasını istiyorum" demek
3. **Effect Size (Cohen's d):** Yakalamak istediğin etkinin büyüklüğü (Konu 51'de öğrendik) küçük bir etki mi, büyük bir etki mi yakalamak istiyoruz?
4. **Örneklem Büyüklüğü (n):** Kaç kişi/gözlem gerektiği

## Genel Mantık (Trade-off'lar)
- **Daha küçük bir etkiyi yakalamak istiyorsak** → daha **büyük** bir örneklem gerekir (küçük etkiler, "gürültü" içinde kaybolmaya daha müsaittir)
- **Daha yüksek bir Power (örn: %90) istiyorsak** → yine daha **büyük** bir örneklem gerekir (daha güvenilir/emin olmak istiyorsak, daha fazla veri lazım)
- **α'yı küçültürsek** (daha katı bir eşik) → yine daha **büyük** bir örneklem gerekir.

## Pratik Kullanım
Python'da `statsmodels` kütüphanesinin `TTestIndPower` sınıfı, bu dört değişkenden istediğini hesaplamana yardım eder — en yaygın kullanım, "Effect Size + α + Power biliniyorken, gereken örneklem büyüklüğünü (n) bulmak."

## Neden Senin Alanın İçin Kritik?
A/B testi tasarlarken, "testi ne kadar süre çalıştırmalıyım / kaç kullanıcıya ihtiyacım var" sorusunun **bilimsel cevabı** budur. Yetersiz örneklemle erken karar vermek (ya da gereğinden fazla kullanıcıyı gereksiz yere teste sokmak) hem yanlış sonuçlara hem kaynak israfına yol açar.

In [5]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

# α=0.05, Power=0.80 sabit, farklı etki büyüklüklerinde gereken örneklem büyüklüğü
for d, isim in [(0.2, "Küçük etki"), (0.5, "Orta etki"), (0.8, "Büyük etki"), (2.84, "Genç-yaşlı örneğimiz")]:
    n = analysis.solve_power(effect_size=d, alpha=0.05, power=0.80, alternative='two-sided')
    print(f"{isim} (d={d}): grup başına yaklaşık {n:.1f} kişi gerekiyor")

Küçük etki (d=0.2): grup başına yaklaşık 393.4 kişi gerekiyor
Orta etki (d=0.5): grup başına yaklaşık 63.8 kişi gerekiyor
Büyük etki (d=0.8): grup başına yaklaşık 25.5 kişi gerekiyor
Genç-yaşlı örneğimiz (d=2.84): grup başına yaklaşık 3.2 kişi gerekiyor


1. Eğer küçük bir etkiyi (d=0.2), %5 yanılma payıyla (α=0.05) ve %80 olasılıkla yakalamak (Power=0.80) istiyorsam, her grup için yaklaşık 394 kişiye ihtiyacım vardır.
2. Eğer orta düzeyde bir etkiyi (d=0.5), %5 yanılma payıyla (α=0.05) ve %80 olasılıkla yakalamak (Power=0.80) istiyorsam, her grup için yaklaşık 63.8 kişiye ihtiyacım vardır.
3. Eğer yüksek düzeyde bir etkiyi (d=0.8), %5 yanılma payıyla (α=0.05) ve %80 
olasılıkla yakalamak (Power=0.80) istiyorsam, her grup için yaklaşık 25.5 kişiye ihtiyacım vardır.

### Sonuç
Etki düzeyi arttıkça (farkedilebilirlik) daha az insana ihtiyaç duyuyoruz.